In [ ]:
# CELL 1: Install dependencies (uncomment if needed)
# !pip install qiskit qiskit-aer matplotlib numpy


In [ ]:
# CELL 2: Import required libraries
import numpy as np
import math
from fractions import Fraction
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator


# Shor's Algorithm (Educational Demo for \(N=15\))

Shor's algorithm is a quantum algorithm that factors integers by reducing factoring to a **period-finding** problem.  
Classically, factoring large integers is believed to be hard, and this hardness underpins cryptosystems such as RSA.  

The core quantum advantage comes from estimating the period \(r\) of the function:
\[
f(x) = a^x \bmod N
\]
Once \(r\) is found, we can often extract non-trivial factors of \(N\) using greatest common divisors.

In this notebook, we demonstrate a full educational version of Shor's algorithm to factor:
\[
N = 15
\]
with base \(a=2\), using a **manually implemented QFT** (no `QFT`/`QFTGate`).


In [ ]:
# CELL 4: Manual QFT implementation
def qft_manual(qc, n):
    """Apply the Quantum Fourier Transform to the first n qubits of qc manually."""
    for j in range(n):
        qc.h(j)
        for k in range(j + 1, n):
            angle = np.pi / (2 ** (k - j))
            qc.cp(angle, k, j)
    for i in range(n // 2):
        qc.swap(i, n - i - 1)


In [ ]:
# CELL 5: Manual inverse QFT implementation
def iqft_manual(qc, n):
    """Apply the inverse QFT to the first n qubits of qc manually."""
    for i in range(n // 2):
        qc.swap(i, n - i - 1)
    for j in reversed(range(n)):
        for k in reversed(range(j + 1, n)):
            angle = -np.pi / (2 ** (k - j))
            qc.cp(angle, k, j)
        qc.h(j)


In [ ]:
# CELL 6: Classical helper functions
def gcd(a, b):
    """Compute gcd(a, b) using Euclid's algorithm."""
    while b != 0:
        a, b = b, a % b
    return abs(a)

def continued_fraction_phase_estimation(phase, max_denominator):
    """Estimate phase as s/r using continued fractions."""
    frac = Fraction(phase).limit_denominator(max_denominator)
    return frac.numerator, frac.denominator


In [ ]:
# CELL 7: Modular exponentiation for a=2, N=15
def c_amod15(qc, control, work, power):
    """Apply controlled multiplication by (2^power) mod 15 on work register."""
    repetitions = power % 4
    for _ in range(repetitions):
        qc.cswap(control, work[0], work[1])
        qc.cswap(control, work[1], work[2])
        qc.cswap(control, work[2], work[3])
    if repetitions in (1, 3):
        qc.cx(control, work[0])
        qc.cx(control, work[1])
        qc.cx(control, work[2])
        qc.cx(control, work[3])


In [ ]:
# CELL 8: Build full Shor circuit
N = 15
a = 2
n_count = 4

qc = QuantumCircuit(n_count + 4, n_count)
count_qubits = list(range(n_count))
work_qubits = list(range(n_count, n_count + 4))

# Initialize work register to |1>
qc.x(work_qubits[0])

# Put counting register into superposition
for q in count_qubits:
    qc.h(q)

# Controlled modular exponentiation blocks U^(2^j)
for j in range(n_count):
    power = 2 ** j
    c_amod15(qc, count_qubits[j], work_qubits, power)

# Inverse QFT on counting qubits
iqft_manual(qc, n_count)


In [ ]:
# CELL 9: Measure counting register
qc.measure(range(n_count), range(n_count))

# Draw the full circuit
qc.draw('mpl')


In [ ]:
# CELL 10: Run the circuit on AerSimulator
backend = AerSimulator()
tqc = transpile(qc, backend)
job = backend.run(tqc, shots=4096)
result = job.result()
counts = result.get_counts()

print('Measurement counts:')
print(counts)


In [ ]:
# CELL 11: Plot measurement histogram using matplotlib
bitstrings = list(counts.keys())
frequencies = [counts[b] for b in bitstrings]

plt.figure(figsize=(8, 4))
plt.bar(bitstrings, frequencies)
plt.title('Shor period-finding outcomes (count register)')
plt.xlabel('Measured bitstring')
plt.ylabel('Counts')
plt.grid(axis='y', alpha=0.3)
plt.show()


In [ ]:
# CELL 12: Classical post-processing to recover factors
measured = sorted(counts.items(), key=lambda x: x[1], reverse=True)
print('Top outcomes and estimated periods:')

candidate_rs = set()
for bitstring, freq in measured[:8]:
    value = int(bitstring, 2)
    phase = value / (2 ** n_count)
    s, r = continued_fraction_phase_estimation(phase, N)
    if r > 0:
        candidate_rs.add(r)
    print(f'bitstring={bitstring}, count={freq}, phase={phase:.4f}, approx={s}/{r}')

found_factors = set()
for r in sorted(candidate_rs):
    if r % 2 != 0:
        continue
    ar2 = pow(a, r // 2, N)
    if ar2 == N - 1:
        continue

    f1 = gcd(pow(a, r // 2) - 1, N)
    f2 = gcd(pow(a, r // 2) + 1, N)

    if 1 < f1 < N:
        found_factors.add(f1)
    if 1 < f2 < N:
        found_factors.add(f2)

print('\nNon-trivial factors found:')
print(found_factors)
if found_factors == {3, 5}:
    print('Success: factors are 3 and 5.')
else:
    print('Probabilistic run: rerun if needed. Expected factors: 3 and 5.')
